In [2]:

!pip install pandas numpy scikit-learn matplotlib seaborn


   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.4 MB 8.2 MB/s eta 0:00:01
   ------------- -------------------------- 2.9/8.4 MB 6.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.4 MB 6.5 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.4 MB 6.5 MB/s eta 0:00:01
   --------------------------------- ------ 7.1/8.4 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 6.6 MB/s  0:00:01
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.4 MB 7.6 MB/s eta 0:00:05
   --- ------------------------------------ 3.4/37.4 MB 8.1 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/37.4 MB 7.5 MB/s eta 0:00:05
   ----- ---------------------------------- 5.5/37.4 MB 6.7 MB/s eta 0:00:05
   ------- -------------------------------- 7.1/37.4 MB 6.7 MB/s eta 0:00:05
   -------- -----------


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import necessary data manipulation, machine learning, and visualization libraries 
import pandas as pd 
import numpy as np 
import datetime as dt 
from sklearn.preprocessing import StandardScaler 
from sklearn.cluster import KMeans 
import matplotlib.pyplot as plt 
import seaborn as sns 
# Set global visualization themes 
sns.set_theme(style="whitegrid") 
print("Environment successfully configured!") 

Environment successfully configured!


In [6]:
# Set random seed for reproducibility 
np.random.seed(42) 
n_transactions = 15000 
start_date = dt.datetime(2025, 1, 1) 

# Generate synthetic transaction attributes 
customer_ids = [f"CUST_{np.random.randint(1000, 2000)}" for _ in range(n_transactions)] 
transaction_ids = [f"TXN_{str(i).zfill(6)}" for i in range(1, n_transactions + 1)] 
random_days = np.random.randint(0, 365, size=n_transactions) 
transaction_dates = [start_date + dt.timedelta(days=int(d)) for d in random_days] 
order_values = np.round(np.random.exponential(scale=75, size=n_transactions) + 10, 2) 

# Create DataFrame 
raw_df = pd.DataFrame({ 
'TransactionID': transaction_ids, 
'CustomerID': customer_ids, 
'TransactionDate': transaction_dates, 
'OrderValue': order_values 
}) 

# Save raw logs and inspect structure 
raw_df.to_csv('raw_customer_transactions.csv', index=False) 
print(f"Dataset successfully created with {len(raw_df)} transaction logs across {raw_df['CustomerID'].nunique()} unique customers!") 
raw_df.head() 

Dataset successfully created with 15000 transaction logs across 1000 unique customers!


,TransactionID,CustomerID,TransactionDate,OrderValue
0,TXN_000001,CUST_1102,2025-05-11,53.44
1,TXN_000002,CUST_1435,2025-03-08,318.28
2,TXN_000003,CUST_1860,2025-08-16,73.14
3,TXN_000004,CUST_1270,2025-06-12,27.00
4,TXN_000005,CUST_1106,2025-01-15,35.98


In [7]:
# Define snapshot date (1 day after the latest transaction date in the dataset) 
snapshot_date = raw_df['TransactionDate'].max() + dt.timedelta(days=1) 

# Group by CustomerID and compute RFM metrics 
rfm_df = raw_df.groupby('CustomerID').agg({ 
'TransactionDate': lambda x: (snapshot_date - x.max()).days,  # Recency 
'TransactionID': 'count',                                    # Frequency 
'OrderValue': 'sum'                                          # Monetary 
}).reset_index() 

# Rename columns to standard RFM nomenclature 
rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary'] 

print(f"RFM aggregated for {len(rfm_df)} customers.") 

RFM aggregated for 1000 customers.


In [8]:
# Select numeric features for clustering 
rfm_features = rfm_df[['Recency', 'Frequency', 'Monetary']] 

# Initialize and fit StandardScaler 
scaler = StandardScaler() 
rfm_scaled = scaler.fit_transform(rfm_features) 

# Convert scaled array back to DataFrame for verification 
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled']) 
rfm_scaled_df.head() 

,Recency_Scaled,Frequency_Scaled,Monetary_Scaled
0,-0.404689,1.004151,0.700345
1,0.604721,0.000000,-0.504873
2,-0.614983,-0.753113,-0.093174
3,0.646779,-1.255188,-0.636782
4,0.226192,-1.255188,-0.510862


In [9]:
# Train K-Means model 
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10) 
rfm_df['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Inspect mean values for each cluster to evaluate performance 
cluster_summary = rfm_df.groupby('Cluster').agg({ 
'Recency': 'mean', 
'Frequency': 'mean', 
'Monetary': ['mean', 'count'] 
}).round(2) 

print("Cluster Summary (Mean Values):") 
print(cluster_summary) 

Cluster Summary (Mean Values):
        Recency Frequency Monetary      
           mean      mean     mean count
Cluster                                 
0         17.77     16.01  1356.02   430
1         20.62     11.42   885.01   295
2         78.38     11.85  1015.82   104
3         16.06     20.56  1938.18   171


In [10]:
# Define persona mapping based on cluster characteristics 
# (Low Recency + High Frequency + High Monetary = Champions) 
def assign_persona(row): 
    if row['Cluster'] == 0: 
        return 'At-Risk / Lost' 
    elif row['Cluster'] == 1: 
        return 'Champions' 
    elif row['Cluster'] == 2: 
        return 'Loyal Customers' 
    else: 
        return 'Recent / New Visitors' 
        
rfm_df['Customer_Persona'] = rfm_df.apply(assign_persona, axis=1) 

# Export the labeled dataset for Tableau ingestion 
rfm_df.to_csv('rfm_segmented_customers.csv', index=False) 
print("Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!") 
rfm_df.head()

Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!


,CustomerID,Recency,Frequency,Monetary,Cluster,Customer_Persona
0,CUST_1000,15,19,1586.42,3,Recent / New Visitors
1,CUST_1001,39,15,1061.24,0,At-Risk / Lost
2,CUST_1002,10,12,1240.64,1,Champions
3,CUST_1003,40,10,1003.76,1,Champions
4,CUST_1004,30,10,1058.63,1,Champions
